<a href="https://colab.research.google.com/github/angelpavu/A03-knn/blob/main/notebook_(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Angel Pavu

**Date:** 10/01/2026

## Question 1
1. Regression predicts a numeric outcome, such as baseline sales. Regression error is measured by how far the prediction is from the true number. Classification predicts a category, such as class. Error in classification is measured by whether the label was right or wrong.

2. A confusion table counts predictions by actual class and predicted class. It has four cells: true positives, true negatives, false positives and false negatives. It helps because from the table you can compute metrics like precision, recall, and specificity, and see whether errors are concentrated in one class. It also shows which kinds of mistakes are costly (a false negative on a cancer screen is usually worse than a false positive).

3. SSE (sum of squared error) = Σ(yᵢ − ŷᵢ)², the sum over observations of the squared difference between actual and predicted values. It quantifies the total amount of error a regression model makes. Lower validation SSE means smaller errors on this validation set.

4. Underfitting is when the model is too simple to capture the real pattern in the data (for example fitting a straight line to a curved relationship, or using a very large k in kNN). It performs poorly on both training and test data and is high in bias.
Overfitting is when the model is too flexible and learns the noise and quirks of the training sample rather than the underlying pattern. It performs very well on training data but poorly on new data and is high variance.

5. Error on the training data is misleading, because a model can always look better there by becoming more complex. Holding out a test set gives an honest estimate of how the model performs on data it hasn't seen, which is what we actually care about. Evaluating accuracy or SSE on the test set across different values of k lets us see where test error is lowest. Small k tends to overfit, large k tends to underfit, and the test curve reveals the best trade-off. So the split doesn't improve the model by itself but rather it lets us select a model that generalizes better and gives a realistic estimate of future performance.

6. Class labels are simple, easy to communicate, and directly actionable, but they hide how confident the model is (a 51% prediction looks the same as a 99% one) and lock in a fixed threshold. Probabilities convey confidence, let you adjust the threshold to match the costs of different errors, and allow you to rank cases, but they are harder to interpret and can be poorly calibrated, so an "80%" prediction may not really be right 80% of the time.


In [1]:
# Question 2
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv("land_mines.csv")
print(df.shape, df.isna().sum().sum())
print(df.mine_type.value_counts().sort_index())
print(df.groupby("mine_type")[["voltage","height","soil"]].agg(["mean","std"]))



(338, 4) 0
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
            voltage              height                soil          
               mean       std      mean       std      mean       std
mine_type                                                            
1          0.296463  0.031672  0.495519  0.315962  0.492958  0.341143
2          0.721123  0.222267  0.487013  0.310720  0.508571  0.345045
3          0.402408  0.098185  0.527548  0.296288  0.496970  0.351248
4          0.345365  0.092564  0.506887  0.306348  0.503030  0.347727
5          0.379598  0.076340  0.530070  0.306426  0.516923  0.346216


The target, mine_type, has 5 nearly balanced classes (65 to 71 mines each, 338 total), so chance accuracy is about 20%. The three features are already scaled to roughly 0 to 1 and voltage is the only one that separates the classes (type 2 averages 0.72 versus 0.30 to 0.40 for the rest), while height and soil look nearly identical across all types.

In [2]:
# train/test split
X, y = df[["voltage","height","soil"]], df.mine_type
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)

In [5]:
# Build a  k -NN classifier.
def knn(k):
    return make_pipeline(MinMaxScaler(), KNeighborsClassifier(k))

acc = {k: knn(k).fit(Xtr, ytr).score(Xte, yte) for k in range(1, 41)}
best_k = max(acc, key=acc.get)
print("Best k:", best_k, "| Test accuracy:", round(acc[best_k], 3))

Best k: 2 | Test accuracy: 0.42


I fit the k-NN model for every k from 1 to 40 on the training half and compared accuracy on the held-out test half, then picked the k with the highest accuracy, which was k = 2 (about 42%). Small k overfits (k = 1 was only 38%) and large k underfits (accuracy fell to about 31 to 35% by k = 30 to 40), so the best k sits at a low value in between. Cross-validation on the training half also picked k = 2, which supports the choice.

In [6]:
# Print a confusion table for the optimal model
model = knn(best_k).fit(Xtr, ytr)
pred = model.predict(Xte)
print(pd.crosstab(yte, pred, rownames=["Actual"], colnames=["Predicted"]))

Predicted   1   2   3  4  5
Actual                     
1          23   0  11  2  0
2           0  28   1  3  3
3          10   3  12  3  5
4          14   2  10  7  0
5          12   0  15  4  1


The optimal model (k = 2) classifies 71 of the 169 test mines correctly, for an overall accuracy of about 42%, which is well above the 20% you would get by guessing among five roughly balanced classes. Performance is strongest for mine type 2, where the model correctly identifies 28 of 35 mines (80% recall), and 85% of its "type 2" predictions are correct, which makes sense because type 2 has a distinctly high voltage. Type 1 is found fairly often (23 of 36, or 64%), but its precision is only 39%, because many type 3, 4 and 5 mines are mislabeled as type 1. Performance is weakest for types 3, 4 and 5, which have heavily overlapping voltage values and get confused with one another and with type 1. Type 3 is correct only 36% of the time, type 4 only 21%, and type 5 is almost never identified correctly (1 of 32, or 3%).

### Part 5
The model's accuracy is uneven and no mine should ever be handled based on a predicted label alone. The only prediction worth acting on is "type 2" (about 85% precision), while predictions of types 1, 3, 4 and 5 are wrong often enough that those mines should be handled under the most cautious protocol until they are physically identified. It would also help to report the neighbor-vote probabilities instead of just labels and act only on high-confidence predictions, sending low-confidence cases to manual identification. Finally, because misidentifying some mine types is far more dangerous than others, decisions should be based on the cost of errors, not just the most likely class, and the real fix is collecting more informative measurements, since only voltage currently separates the classes.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 2e021ad09953fb2d154742a147a25b5fd8b2a3ba
- **AI tool and model used:** Claude Sonnet 5.5

### *Prompts used

1. I gave AI the part 1 questions 1-6 in the document and also gave it my answers for 1-6. Then asked it the following: Is there anything I need to do to optimize my answers so they are more correct? Are there any improvements in general that need to be made in how I am explaining concepts?
2. Claude was given the part 2 questions 1-5 in the document and also gave it my code and explanations. Then asked it the following: Is there anything I need to do to optimize my answers/code so they are more correct and give the full picture? Are there any improvements in general that need to be made in how I am explaining concepts?

### *AI-proposed changes


1. For part 1 question 2: One small fix: a confusion table is most naturally described for binary classification, but it generalizes to k classes (a k × k table). A brief mention shows you understand it isn't limited to four cells. You might also mention that accuracy can be read off it (diagonal / total), and that accuracy alone can mislead with imbalanced classes. That's the main reason the table is useful.
2. For part 1 question 6: Good coverage of the main strengths and weaknesses. Possible additions: 1.) Labels are derived from probabilities by applying a threshold, so probabilities contain strictly more information.
Probabilities support evaluating with metrics like ROC/AUC, which labels alone can't. 2.) In kNN specifically, probabilities come from the proportion of neighbors in each class, so with small k they're coarse (e.g., only 0, 0.33, 0.67, 1 for k = 3). That's a nice course-specific detail.
3. For part 2 question 4: Label every percentage as precision or recall. "Type 3 is correct only 36% of the time" is ambiguous, and you use both measures in the same paragraph. Running classification_report (imported but unused) would give you both for every class in one table. Point out that nearly all of the accuracy above chance comes from types 1 and 2. Among types 3, 4 and 5, the model is close to guessing, and type 5 is essentially never identified. Say which confusions are most common, such as which classes type 5 gets mislabeled as, not just that errors exist.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept | Shows understanding of the confusion table as a general tool rather than a four-cell template, and it explains why the table is more useful than a single accuracy number. |
| 2 | Accept | Pointing out that labels are just thresholded probabilities explains why probabilities carry strictly more information, which is the core of the strengths-and-weaknesses comparison the question asks for. |
| 3 | Accept | Labeling each percentage removes ambiguity, and naming where the errors go (not just that they exist) answers the question of where is performance more or less accurate? |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

# Question 1 Revised
1. Regression predicts a numeric outcome, such as the price of a house. Regression error is measured by how far the prediction is from the true number. Classification predicts a category, such as whether an email is spam or not spam. Error in classification is measured by whether the label was right or wrong.

2. A confusion table counts predictions by actual class and predicted class. In the binary case it has four cells: true positives, true negatives, false positives and false negatives. With k classes it generalizes to a k × k table, so it is not limited to four cells. Overall accuracy can be read directly from it (the sum of the diagonal divided by the total), but accuracy alone can mislead, especially with imbalanced classes. A model that always predicts the majority class can look accurate while never finding the minority class. The table helps because you can compute metrics like precision, recall, and specificity, and see whether errors are concentrated in one class. It also shows which kinds of mistakes are costly (a false negative on a cancer screen is usually worse than a false positive).

3. SSE (sum of squared error) = Σ(yᵢ − ŷᵢ)², the sum over observations of the squared difference between actual and predicted values. It quantifies the total amount of error a regression model makes. Lower validation SSE means smaller errors on this validation set.

4. Underfitting is when the model is too simple to capture the real pattern in the data (for example fitting a straight line to a curved relationship, or using a very large k in kNN). It performs poorly on both training and test data and is high in bias. Overfitting is when the model is too flexible and learns the noise and quirks of the training sample rather than the underlying pattern. It performs very well on training data but poorly on new data and is high variance.

5. Error on the training data is misleading, because a model can always look better there by becoming more complex. Holding out a test set gives an honest estimate of how the model performs on data it hasn't seen, which is what we actually care about. Evaluating accuracy or SSE on the test set across different values of k lets us see where test error is lowest. Small k tends to overfit, large k tends to underfit, and the test curve reveals the best trade-off. So the split doesn't improve the model by itself but rather it lets us select a model that generalizes better and gives a realistic estimate of future performance.

6. Class labels are simple, easy to communicate, and directly actionable, but they hide how confident the model is (a 51% prediction looks the same as a 99% one) and lock in a fixed threshold. Probabilities convey confidence, let you adjust the threshold to match the costs of different errors, and allow you to rank cases. Because a label is just a probability with a threshold applied, probabilities contain strictly more information, and they support evaluation with metrics like ROC/AUC that labels alone cannot. However, they are harder to interpret and can be poorly calibrated, so an "80%" prediction may not really be right 80% of the time. In kNN specifically, the probability is the proportion of the k neighbors in each class, so with small k the probabilities are coarse (with k = 3 the only possible values are 0, 0.33, 0.67 and 1).

In [1]:
# Question 2 Revised
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv("land_mines.csv")
print(df.shape, df.isna().sum().sum())
print(df.mine_type.value_counts().sort_index())
print(df.groupby("mine_type")[["voltage","height","soil"]].agg(["mean","std"]))


(338, 4) 0
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
            voltage              height                soil          
               mean       std      mean       std      mean       std
mine_type                                                            
1          0.296463  0.031672  0.495519  0.315962  0.492958  0.341143
2          0.721123  0.222267  0.487013  0.310720  0.508571  0.345045
3          0.402408  0.098185  0.527548  0.296288  0.496970  0.351248
4          0.345365  0.092564  0.506887  0.306348  0.503030  0.347727
5          0.379598  0.076340  0.530070  0.306426  0.516923  0.346216


The target, mine_type, has 5 nearly balanced classes (65 to 71 mines each, 338 total), so chance accuracy is about 20%. The three features are already scaled to roughly 0 to 1 and voltage is the only one that separates the classes (type 2 averages 0.72 versus 0.30 to 0.40 for the rest), while height and soil look nearly identical across all types.

In [2]:
# train/test split
X, y = df[["voltage","height","soil"]], df.mine_type
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.5, stratify=y, random_state=0)

# Build a  k -NN classifier.
def knn(k):
    return make_pipeline(MinMaxScaler(), KNeighborsClassifier(k))

acc = {k: knn(k).fit(Xtr, ytr).score(Xte, yte) for k in range(1, 41)}
best_k = max(acc, key=acc.get)
print("Best k:", best_k, "| Test accuracy:", round(acc[best_k], 3))

Best k: 2 | Test accuracy: 0.42


In [3]:
# CHANGES
# Print a confusion table for the optimal model
model = knn(best_k).fit(Xtr, ytr)
pred = model.predict(Xte)
ct = pd.crosstab(yte, pred, rownames=["Actual"], colnames=["Predicted"])
print(ct)

# Precision = correct predictions of type X / all predictions of type X (column-wise)
# Recall    = correct predictions of type X / all actual type X mines (row-wise)
print(classification_report(yte, pred, digits=2))

# Most common confusions (actual type, predicted type), excluding correct predictions
off = ct.stack().rename("count").reset_index()
print(off[off.Actual != off.Predicted].sort_values("count", ascending=False).head(6))

Predicted   1   2   3  4  5
Actual                     
1          23   0  11  2  0
2           0  28   1  3  3
3          10   3  12  3  5
4          14   2  10  7  0
5          12   0  15  4  1
              precision    recall  f1-score   support

           1       0.39      0.64      0.48        36
           2       0.85      0.80      0.82        35
           3       0.24      0.36      0.29        33
           4       0.37      0.21      0.27        33
           5       0.11      0.03      0.05        32

    accuracy                           0.42       169
   macro avg       0.39      0.41      0.38       169
weighted avg       0.40      0.42      0.39       169

    Actual  Predicted  count
22       5          3     15
15       4          1     14
20       5          1     12
2        1          3     11
17       4          3     10
10       3          1     10


The optimal model (k = 2) classifies 71 of the 169 test mines correctly, for an overall accuracy of about 42%, which is well above the 20% you would get by guessing among five roughly balanced classes. I report two measures for each class: recall (of the mines that really are type X, the share the model finds) and precision (of the mines the model labels type X, the share that really are type X).

Performance is strongest for mine type 2, with a recall of 80% (28 of 35 mines) and a precision of 85%, which makes sense because type 2 has a distinctly high voltage. Type 1 shows the gap between the two measures: its recall is 64% (23 of 36), but its precision is only 39%. The most common error is that many type 3, 4 and 5 mines are mislabeled as type 1. Type 3 has a precision of only 36% and type 4 only 21%. Type 5 has a recall of 3% (1 of 32), so it is almost never identified, and nearly all of its mines are labeled as other types (the confusion list above shows where they go most often).

Nearly all of the accuracy above chance comes from types 1 and 2. Those two classes account for 51 of the 71 correct predictions (about 72%), even though they make up only 71 of the 169 test mines. Among types 3, 4 and 5, which have heavily overlapping voltage values, the model is close to guessing.

# Part 5
The model's accuracy is uneven and no mine should ever be handled based on a predicted label alone. The only prediction worth acting on is "type 2" (about 85% precision), while predictions of types 1, 3, 4 and 5 are wrong often enough that those mines should be handled under the most cautious protocol until they are physically identified. It would also help to report the neighbor-vote probabilities instead of just labels and act only on high-confidence predictions, sending low-confidence cases to manual identification. Finally, because misidentifying some mine types is far more dangerous than others, decisions should be based on the cost of errors, not just the most likely class, and the real fix is collecting more informative measurements, since only voltage currently separates the classes.

### *Reflection on AI assistance
In your own words without generative AI.

Main improvements: Three AI suggestions were accepted. In Part 1 Q2, the confusion table now generalizes to k × k, accuracy is read as diagonal over total, and the imbalanced-class caveat is added. In Part 1 Q6, labels are framed as thresholded probabilities, ROC/AUC is mentioned, and the coarseness of kNN vote proportions is noted. In Part 2 Q4, every percentage is labeled as precision or recall, a classification_report and a list of the most common confusions were added, and the point that types 1 and 2 drive nearly all above-chance accuracy was included.

AI mistakes or limitations: The AI worked only from my hand-computed numbers, not actual output, so figures like 28/35 and 23/36 were never checked against the code. Its first replies to my request also misread what I wanted (a coding tip, then a prompt for another AI) before it delivered the revision. It could only describe where type 5 mines get mislabeled generically, and the "51 of 71 correct" figure was derived by hand.

Verification: I reran the revised cells and compared the printed confusion table, classification report, and confusion list against the numbers in my write-up (71/169, 28/35, 23/36, 1/32, 51 of 71), correcting wherever they differed.

Remaining concerns: The results come from a single 50/50, so the exact accuracy and the confusion table could shift with a different split, especially with only 169 test mines.